In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import warnings 
warnings.filterwarnings("ignore")

## Reviewer Run Tags

- `[REVIEWER-RUNNABLE]`: can be run from files included in this repository, such as CSV, descriptor pickle, ASE database, Parquet, or existing figure inputs. No raw Gaussian working directory is required.
- `[RAW-GAUSSIAN/E:/work]`: depends on raw calculation artifacts, Gaussian logs, Mol files, ORCA/Gaussian input-output folders, or the external raw calculation root configured by `BORYLXAT_RAW_CALC_ROOT` (default `E:/work/B_Cl_Nu`). These cells document provenance but are not required for routine review reruns.
- `[OPTIONAL-DESCRIPTOR-GENERATION]`: regenerates descriptors from the released database. Reviewers can skip it and use the pre-extracted descriptor pickle files to save time.


In [ ]:
from DFTStructureGenerator.draw import plot_scatter_fit, plot_distribution, plot_panel
from DFTStructureGenerator.Build_DataBase import build_databases
from DFTStructureGenerator.Tool import get_bond_angle
from tqdm import tqdm
from rdkit import Chem
from DFTStructureGenerator.logfile_process import Logfile
from DFTStructureGenerator import borane_xat_workflow
import os
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from ase import Atoms
from ase.db import connect
from DFTStructureGenerator.project_paths import CSV_DIR, TS_DATA_DIR, raw_calc_file


# Build Database [RAW-GAUSSIAN/E:/work]

This provenance section reconstructs `combined_data` from raw Mol files, Gaussian optimization/SPE logs, wfn-derived charge files, IRC logs, and the external raw calculation root. Reviewers can skip it when using the released `BorylXAT-DB.db` and `BorylXAT-DB.parquet`.


In [ ]:
duplicate_N_id = [9, 43, 285, 310, 314, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 372, 375, 376]



Borane_df = pd.read_csv(CSV_DIR / "reactants_B.csv")
nu_df = pd.read_csv(CSV_DIR / "reactants_N.csv").dropna(subset=['G_energy_r'])
Cl_df = pd.read_csv(CSV_DIR / "reactants_Cl.csv")
Borane_nu_df = pd.read_csv(CSV_DIR / "reactants_B_N.csv")
reaction_df = pd.read_csv(TS_DATA_DIR / "Borane_all.csv")


# raise NameError
combined_data = {}
temp_combined_data = {}

# ======================== B monomer (open shell) ========================
for row_id, row in Borane_df.iterrows():
    idx, conf_idxs_r, G = row['Index'], int(row['conf_idxs_r']), row['G_energy_r']
    mol = Chem.MolFromMolFile(raw_calc_file("Data", "Mols", f"B_{idx:05}_r.mol"), removeHs=False)
    [atom.SetAtomMapNum(atom.GetIdx()) for atom in mol.GetAtoms()]
    smiles = Chem.MolToSmiles(mol)
    log_file = raw_calc_file("Data", "GS_OPT", "B_single", f"B_{idx:05}_r_{conf_idxs_r:04}.log")
    log = Logfile(log_file)
    atoms = Atoms(symbols=log.symbol_list, positions=log.running_positions[-1])
    # Descriptor: SPE log
    spe_log_file = log_file.replace("GS_OPT", "GS_SPE")
    spe_log = Logfile(spe_log_file)
    dipole_moment = spe_log.get_dipole()
    _, spin_densities = spe_log.read_charge_spin_density()
    orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])[:2]  # Open the shell and take the first 2
    combined_data[f'B_{idx:05}'] = {
        "smiles": smiles, "atoms": atoms, "gibbs_hartree": G,
        "dipole_moment": dipole_moment,
        "spin_densities": spin_densities,
        "homo_energy_kcal": orbital_energies[0],
        "lumo_energy_kcal": orbital_energies[1],
    }

# ======================== LB / Nu monomer (closed shell) ========================
for row_id, row in nu_df.iterrows():
    idx, conf_idxs_r, G = row['Index'], int(row['conf_idxs_r']), row['G_energy_r']
    mol = Chem.MolFromMolFile(raw_calc_file("Data", "Mols", f"N_{idx:05}_r.mol"), removeHs=False)
    [atom.SetAtomMapNum(atom.GetIdx()) for atom in mol.GetAtoms()]
    smiles = Chem.MolToSmiles(mol)
    log_file = raw_calc_file("Data", "GS_OPT", "N_single", f"N_{idx:05}_r_{conf_idxs_r:04}.log")
    log = Logfile(log_file)
    atoms = Atoms(symbols=log.symbol_list, positions=log.running_positions[-1])
    # Descriptor: SPE log
    spe_log_file = log_file.replace("GS_OPT", "GS_SPE")
    spe_log = Logfile(spe_log_file)
    dipole_moment = spe_log.get_dipole()
    orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])  # Closed shell layer returns 2
    combined_data[f'LB_{idx:05}'] = {
        "smiles": smiles, "atoms": atoms, "gibbs_hartree": G,
        "dipole_moment": dipole_moment,
        "homo_energy_kcal": orbital_energies[0],
        "lumo_energy_kcal": orbital_energies[1],
    }

# ======================== Cl substrate (r closed shell) / C radical (p open shell) ========================
for row_id, row in Cl_df.iterrows():
    idx, conf_idxs_r, conf_idxs_p, G_r, G_p = row['Index'], int(row['conf_idxs_r']), int(row['conf_idxs_p']), row['G_energy_r'], row['G_energy_p']
    for conf_id, state, G in [[conf_idxs_r, 'r', G_r], [conf_idxs_p, 'p', G_p]]:
        mol = Chem.MolFromMolFile(raw_calc_file("Data", "Mols", f"Cl_{idx:05}_{state}.mol"), removeHs=False)
        [atom.SetAtomMapNum(atom.GetIdx()) for atom in mol.GetAtoms()]
        smiles = Chem.MolToSmiles(mol)
        log_file = raw_calc_file("Data", "GS_OPT", f"Cl_{state}", f"Cl_{idx:05}_{state}_{conf_id:04}.log")
        log = Logfile(log_file)
        atoms = Atoms(symbols=log.symbol_list, positions=log.running_positions[-1])
        # Descriptor: SPE log
        spe_log_file = log_file.replace("GS_OPT", "GS_SPE")
        spe_log = Logfile(spe_log_file)
        chg_file = spe_log_file.replace(".log", ".chg").replace("GS_SPE", "GS_SPE_wfn")
        hirshfeld_charges = borane_xat_workflow.FormatConverter.read_chg_file(chg_file)['charge'].to_list()
        dipole_moment = spe_log.get_dipole()
        entry = {
            "smiles": smiles, "atoms": atoms, "gibbs_hartree": G,
            "hirshfeld_charges": hirshfeld_charges,
            "dipole_moment": dipole_moment,
        }
        if state == 'p':
            # c_radical: open shell → spin density + orbital [:2]
            _, spin_densities = spe_log.read_charge_spin_density()
            orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])[:2]
            entry["spin_densities"] = spin_densities
        else:
            # Cl_r: closed shell
            orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])
        entry["homo_energy_kcal"] = orbital_energies[0]
        entry["lumo_energy_kcal"] = orbital_energies[1]
        combined_data[f'Cl_{idx:05}_{state}'] = entry

# ======================== TS transition state ========================
BN_smiles = {}

for row_id, row in tqdm(reaction_df.iterrows()):
    B_idx, N_idx, Cl_idx, conf_id, G, N_atom, AAM = row['B_Index'], row['N_Index'], row['Cl_Index'], row['conf_idxs_ts'], row['TS_G'], int(row['N_Atomid']), row['AAM']
    log_file = raw_calc_file("Sum", "TS_needIRC", f"B_{B_idx:05}_Nu_{N_idx:05}_Cl_{Cl_idx:05}.log")
    if not os.path.exists(log_file):
        log_file = raw_calc_file("Sum", "TS_needIRC", f"B_{B_idx:05}_Nu_{N_idx:05}_Cl_{Cl_idx:05}_{conf_id:04}.log")
    
    irc_forward = log_file.replace("TS_needIRC", "IRC_Full").replace(".log", "forward.log")
    irc_reverse = log_file.replace("TS_needIRC", "IRC_Full").replace(".log", "reverse.log")
    log = Logfile(log_file,ignore_print=True)
    first_unreal_freq = log.first_unreal_freq
    first_unreal_freq_matrix = np.array(log.unreal_freq_matrix)
    atoms = Atoms(symbols=log.symbol_list, positions=log.running_positions[-1])

    # IRC endpoint structure
    irc_fwd_positions = None
    irc_rev_positions = None
    if os.path.exists(irc_forward):
        irc_fwd_log = Logfile(irc_forward, ignore_print=True)
        irc_fwd_positions = irc_fwd_log.running_positions[-1].tolist()
    if os.path.exists(irc_reverse):
        irc_rev_log = Logfile(irc_reverse,ignore_print=True)
        irc_rev_positions = irc_rev_log.running_positions[-1].tolist()

    temp_combined_data[f'B_{B_idx:05}_LB_{N_idx:05}_Cl_{Cl_idx:05}'] = {
        "smiles": AAM, "atoms": atoms, "gibbs_hartree": G,
        "imaginary_frequency_cm_1": float(first_unreal_freq),
        "imaginary_freq_displacement": first_unreal_freq_matrix.tolist(),
        "irc_forward_positions": irc_fwd_positions,
        "irc_reverse_positions": irc_rev_positions,
    }
    BN_name = f'{B_idx:05}_{N_idx:05}'
    if N_idx in duplicate_N_id:
        BN_name = f'{B_idx:05}_{N_idx:05}_{N_atom}'

    BN_smiles[f'{BN_name}_r'] = row['AAM'].split(">>")[0].split('.')[0]
    BN_smiles[f'{BN_name}_p'] = row['AAM'].split(">>")[1].split('.')[0]


# ======================== B-LB complex (r open shell / p closed shell) ========================
for row_id, row in tqdm(Borane_nu_df.iterrows()):
    B_idx, N_idx, conf_idxs_r, conf_idxs_p, G_r, G_p, N_atom, B_atom = row['B_Index'], row['N_Index'], int(row['conf_idxs_r']), int(row['conf_idxs_p']), row['G_energy_r'], row['G_energy_p'], int(row['N_Atomid']), int(row['B_Atomid'])
    B_smiles = row['B_smiles'] 
    N_smiles = row['N_smiles']
    react_mol, product_mol = borane_xat_workflow.generate_ts_mol(B_smiles, N_smiles, B_atom, N_atom)
    react_smiles, product_smiles = Chem.MolToSmiles(react_mol), Chem.MolToSmiles(product_mol)
    for conf_id, state, G, smiles in [[conf_idxs_r, 'r', G_r, react_smiles], [conf_idxs_p, 'p', G_p, product_smiles]]:
        BN_name = f'{B_idx:05}_{N_idx:05}'
        if N_idx in duplicate_N_id:
            BN_name = f'{B_idx:05}_{N_idx:05}_{N_atom}'
        # if f'{BN_name}_{state}' not in BN_smiles.keys():
        #     continue
        # smiles = BN_smiles[f'{BN_name}_{state}']
        log_file = raw_calc_file("Data", "GS_OPT", f"B_N_{state}", f"B_{B_idx:05}_Nu_{N_idx:05}_{state}_{conf_id:04}.log")
        if N_idx in duplicate_N_id:
            log_file = raw_calc_file("Data", "GS_OPT", f"B_N_{state}_d", f"B_{B_idx:05}_Nu_{N_idx:05}_Naid_{N_atom:05}_{state}_{conf_id:04}.log")
        if not os.path.exists(log_file):
            print(BN_name, 'Not Found')
        log = Logfile(log_file,ignore_print=True)
        atoms = Atoms(symbols=log.symbol_list, positions=log.running_positions[-1])
        # Descriptor: SPE log
        spe_log_file = log_file.replace("GS_OPT", "GS_SPE")
        spe_log = Logfile(spe_log_file,ignore_print=True)
        chg_file = spe_log_file.replace(".log", ".chg").replace("GS_SPE", "GS_SPE_wfn")
        hirshfeld_charges = borane_xat_workflow.FormatConverter.read_chg_file(chg_file)['charge'].to_list()
        dipole_moment = spe_log.get_dipole()
        entry = {
            "smiles": smiles, "atoms": atoms, "gibbs_hartree": G,
            "hirshfeld_charges": hirshfeld_charges,
            "dipole_moment": dipole_moment,
        }
        if state == 'r':
            # complex_r: open shell → spin density + orbital take[:2]
            _, spin_densities = spe_log.read_charge_spin_density()
            orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])[:2]
            entry["spin_densities"] = spin_densities
        else:
            # complex_p: closed shell
            orbital_energies = spe_log.read_orbit_eng(HOMO_index=[-1], LUMO_index=[0])
        entry["homo_energy_kcal"] = orbital_energies[0]
        entry["lumo_energy_kcal"] = orbital_energies[1]
        combined_data[f'B_{B_idx:05}_LB_{N_idx:05}_{state}'] = entry

combined_data = combined_data | temp_combined_data

In [ ]:
build_databases(combined_data, rewrite=True)

## Inspect Released Database [REVIEWER-RUNNABLE]

These cells read the released ASE database directly and do not require the raw `E:/work` Gaussian calculation folders.


In [21]:
db = connect("BorylXAT-DB.db")
reaction_df = pd.read_csv(TS_DATA_DIR / "Borane_all.csv")

In [22]:
categories = ["B", "LB", "Cl", "complex_r", "complex_p", "ts", "c_radical"]

counts = pd.Series(
    {cat: db.count(category=cat) for cat in categories},
    name="count"
)

print("Total:", db.count())
print(counts)


Total: 50057
B               55
LB             387
Cl             179
complex_r    20010
complex_p    20010
ts            9237
c_radical      179
Name: count, dtype: int64
